# 07 — Field Program (operational notebook)

**This is not an analysis notebook.** It is the canonical field-day companion:
site rationale, embedded maps, checklists, and space to type observations —
built for a laptop or tablet while planning and running a field day.
Quick phone reference: [`docs/field_program.md`](../docs/field_program.md).
Selection methodology: notebook 06. Data: Jan–Jun 2026; no new analysis here.

# Overview

**Purpose of today's investigation:** ground-truth the data's picture of
illegal dumping at three kinds of places — chronic hotspots, typical-Valley
recurring sites, and clean comparison areas — and answer the standing
tracker questions that only a site visit can (RT-01/02/03, RT-07/08/09).
Success = better places understood and better questions asked, not
maximum sites visited.

Safety/etiquette: several sites are encampment-adjacent. Don't photograph
people or tents up close; shoot the curb and context. If someone engages,
lead with curiosity, not clipboard.


## Maps (open before you drive)

Interactive HTML maps in [`field-notes/maps/`](../field-notes/maps/) — open
locally in any browser, or AirDrop/email the files to your phone (each is
<25 KB; tiles load from the network). Every marker popup includes an
*Open in Google Maps* link for turn-by-turn.

- **[Valley route](../field-notes/maps/valley_route.html)** — all 9 Valley
  sites, numbered in visit order and connected
- **[Koreatown pair](../field-notes/maps/koreatown.html)** — the two New
  Hampshire sites with the 300m/800m migration-check rings
- **[Citywide planning](../field-notes/maps/citywide.html)** — all top 10
  sites, for planning future trips (not routing)

Regenerate anytime: `python scripts/build_field_maps.py`

## Route overview

**Route: Reseda → Encino → Van Nuys → Panorama City → NoHo NE → NoHo Arts
→ Studio City → Sherman Oaks.**

Driving bonus, no stops needed: the **Van Nuys Blvd → Ventura Blvd
gradient**. Between stops 5 → 8 and 8 → 9 you cross the sharpest demand
boundary in the Valley (1.60x → 0.59x citywide mix). Note *where* the street
character changes and whether it tracks the NC boundary.

## Suggested One-Day Itinerary

**Morning (8:00–12:00) — West Valley + gradient**
Crews clear piles during the day, so morning shows the truest accumulation.
- 8:00 Reseda: Sherman Way pair (stops 1–2) — walk the block between them
- 9:15 Encino: Yarmouth Ave pocket (stop 3)
- 10:15 Van Nuys: Delano St (+ Erwin St if quick) (stop 4)
- 11:15 Panorama City: Van Nuys Blvd (stop 5)

**Afternoon (12:30–4:30) — East Valley + controls**
- 12:30 NoHo NE: 13036 Sherman Way (stop 6), lunch on Lankershim
- 1:45 NoHo: 5767 Lankershim (stop 7)
- 2:45 Studio City: Arch Dr (stop 8), then drive Ventura Blvd west —
  clean-control pass (Section C #1)
- 3:45 Sherman Oaks: Addison St (stop 9); Ventura corridor pass (C #2)

**Optional stops (energy permitting)**
- **Koreatown pair over the hill** (~30 min from Sherman Oaks via 101):
  4th & New Hampshire + 114 S New Hampshire (A #1 + #6) — the natural
  experiment. Highest-value single detour available.
- Woodland Hills–Warner Center commercial pass (C #3) — better saved for a
  West Valley morning.
- Encino's Newcastle/White Oak pockets if Yarmouth went quickly.

In [1]:
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd

program = pd.read_csv(Path("../data/processed/field_program_nb06.csv"))
A = program[program["section"] == "A_citywide"].reset_index(drop=True)
B = program[program["section"] == "B_valley"].reset_index(drop=True)
bounds = gpd.read_file("../data/raw/la_neighborhood_councils.geojson")
bounds["NAME"] = bounds["NAME"].str.upper().str.strip()

ROUTE_ORDER = ["17808", "17900", "5130", "14102", "9404", "13036", "5767", "4176", "14359"]
routed = pd.concat([B[B["address"].str.startswith(n)] for n in ROUTE_ORDER]).reset_index(drop=True)

def gmaps(r):
    return f"https://www.google.com/maps/search/?api=1&query={r['lat']:.6f},{r['lon']:.6f}"

def popup(r, why_col="why_interesting"):
    return folium.Popup(
        f"<b>{r['address']}</b><br>{r['neighborhood']}<br>"
        f"reports: {r['reports']} | median gap: {r['median_gap_days']} d<br>"
        f"<i>{r[why_col]}</i><br><a href='{gmaps(r)}' target='_blank'>Open in Google Maps</a>",
        max_width=300)

def stop_map(row, others, zoom=16, color="royalblue"):
    """Mini map: current stop highlighted, sibling sites gray for context."""
    m = folium.Map(location=[row["lat"], row["lon"]], zoom_start=zoom, tiles="cartodbpositron")
    for _, o in others.iterrows():
        if o["address"] != row["address"]:
            folium.CircleMarker([o["lat"], o["lon"]], radius=5, color="#999",
                                fill=True, fill_opacity=0.6, tooltip=o["address"]).add_to(m)
    folium.CircleMarker([row["lat"], row["lon"]], radius=12, color=color, weight=3,
                        fill=True, fill_opacity=0.75, popup=popup(row)).add_to(m)
    return m

print(f"{len(A)} citywide sites, {len(B)} Valley sites loaded")

10 citywide sites, 9 Valley sites loaded


## Interactive Valley map — all 9 sites in visit order

In [2]:
m = folium.Map(location=[34.19, -118.44], zoom_start=12, tiles="cartodbpositron")
folium.PolyLine(routed[["lat", "lon"]].values.tolist(), color="royalblue",
                weight=3, opacity=0.6, dash_array="6").add_to(m)
for i, r in routed.iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=13, color="royalblue",
                        fill=True, fill_color="white", fill_opacity=1, popup=popup(r)).add_to(m)
    folium.map.Marker([r["lat"], r["lon"]], icon=folium.DivIcon(
        html=f"<div style='font-size:11px;font-weight:bold;color:royalblue;"
             f"text-align:center;width:26px;margin-left:-13px;margin-top:-7px'>{i+1}</div>")).add_to(m)
m

## Interactive Citywide map — top 10 (trip planning, not routing)

In [3]:
m = folium.Map(location=[34.04, -118.32], zoom_start=11, tiles="cartodbpositron")
for _, r in A.iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=9, color="crimson", weight=2,
                        fill=True, fill_opacity=0.85, popup=popup(r),
                        tooltip=r["address"]).add_to(m)
m

## Interactive Koreatown map — the New Hampshire pair

In [4]:
pair = A[A["address"].str.contains("NEW HAMPSHIRE")]
quiet = pair[pair["why_interesting"].str.contains("QUIET")].iloc[0]
active = pair[~pair["why_interesting"].str.contains("QUIET")].iloc[0]
m = folium.Map(location=[(quiet["lat"] + active["lat"]) / 2,
                         (quiet["lon"] + active["lon"]) / 2],
               zoom_start=16, tiles="cartodbpositron")
for r, color, label in [(quiet, "gray", "WENT QUIET Apr 17 - what changed?"),
                        (active, "crimson", "still active - the control")]:
    folium.CircleMarker([r["lat"], r["lon"]], radius=12, color=color, weight=3,
                        fill=True, fill_opacity=0.7, popup=popup(r),
                        tooltip=f"{r['address']} - {label}").add_to(m)
for radius, dash in [(300, None), (800, "8")]:
    folium.Circle([quiet["lat"], quiet["lon"]], radius=radius, color="#888", weight=1.5,
                  fill=False, dash_array=dash,
                  tooltip=f"{radius}m ring (dumping stayed flat here after the site went quiet)").add_to(m)
m

---

# Valley Stops

In visit order (matches the route map numbering).

## B1. **17808 Sherman Way, Reseda** (53 · ~2.8d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.200306,-118.521248)

**Why we're here:** start of the corridor pair. Walk east from here.

**Known findings (from the data):** 53 reports Jan–Jun · median gap 2.8 d · LSD-serviced 0% · gone-on-arrival 28% · neighborhood: Reseda

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [5]:
stop_map(B[B["address"].str.startswith("17808")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B2. **17900 Sherman Way, Reseda** (78 · ~1.3d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.200925,-118.522869)

**Why we're here:** 90m away. Key question: one continuous condition or two distinct spots? What's between them?

**Known findings (from the data):** 78 reports Jan–Jun · median gap 1.3 d · LSD-serviced 27% · gone-on-arrival 27% · neighborhood: Reseda

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [6]:
stop_map(B[B["address"].str.startswith("17900")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B3. **5130 Yarmouth Ave, Encino** (37 · ~3.5d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.163793,-118.519947)

**Why we're here:** *chronic pocket inside a clean neighborhood.* Just off the Ventura corridor. Why THIS street? (Newcastle Ave & White Oak Ave pockets are 1–2 blocks over if time allows.)

**Known findings (from the data):** 37 reports Jan–Jun · median gap 3.5 d · LSD-serviced 0% · gone-on-arrival 16% · neighborhood: Encino

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [7]:
stop_map(B[B["address"].str.startswith("5130")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B4. **14102 Delano St, Van Nuys** (58 · ~2.1d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.182407,-118.440532)

**Why we're here:** the Valley dumping capital's top site; 31% gone-on-arrival. Erwin St (41 reports) is 3 blocks south if time allows.

**Known findings (from the data):** 58 reports Jan–Jun · median gap 2.1 d · LSD-serviced 0% · gone-on-arrival 31% · neighborhood: Van Nuys

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [8]:
stop_map(B[B["address"].str.startswith("14102")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B5. **9404 Van Nuys Blvd, Panorama City** (35 · ~5.5d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.240833,-118.449612)

**Why we're here:** the *gone-on-arrival anomaly* area: reports here more often end with nothing collected, and closures are slowest. Does the street look scavenged?

**Known findings (from the data):** 35 reports Jan–Jun · median gap 5.5 d · LSD-serviced 0% · gone-on-arrival 23% · neighborhood: Panorama City

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [9]:
stop_map(B[B["address"].str.startswith("9404")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B6. **13036 Sherman Way, North Hollywood (91605)** (46 · ~2.5d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.200450,-118.417118)

**Why we're here:** encampment-adjacent (LSD), eastern Sherman Way. Compare with the Reseda end of the same street from stop 1–2.

**Known findings (from the data):** 46 reports Jan–Jun · median gap 2.5 d · LSD-serviced 100% · gone-on-arrival 20% · neighborhood: North Hollywood

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [10]:
stop_map(B[B["address"].str.startswith("13036")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B7. **5767 Lankershim Blvd, NoHo** (60 · ~1.6d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.175602,-118.381488)

**Why we're here:** encampment-adjacent archetype on the NoHo spine.

**Known findings (from the data):** 60 reports Jan–Jun · median gap 1.6 d · LSD-serviced 100% · gone-on-arrival 12% · neighborhood: North Hollywood

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [11]:
stop_map(B[B["address"].str.startswith("5767")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B8. **4176 Arch Dr, Studio City** (22 · ~6.3d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.143093,-118.372261)

**Why we're here:** *the clean-area ceiling.* This is as bad as Studio City gets. Calibrate: how does the worst spot in a clean neighborhood compare to an average Van Nuys corner?

**Known findings (from the data):** 22 reports Jan–Jun · median gap 6.3 d · LSD-serviced 0% · gone-on-arrival 14% · neighborhood: Studio City

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [12]:
stop_map(B[B["address"].str.startswith("4176")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## B9. **14359 Addison St, Sherman Oaks** (26 · ~3.4d)

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.161741,-118.446466)

**Why we're here:** moderate-area top site; end of loop.

**Known findings (from the data):** 26 reports Jan–Jun · median gap 3.4 d · LSD-serviced 0% · gone-on-arrival 19% · neighborhood: Sherman Oaks

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [13]:
stop_map(B[B["address"].str.startswith("14359")].iloc[0], B)

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

---

# Citywide Stops

## A1. **4th St & New Hampshire Ave — Koreatown** · 115 reports · every ~0.4 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.067373,-118.292947)

**Why we're here:** **THE headline site.** Citywide #1, fully encampment-serviced (LSD), reported ~daily for 3.5 months — then **silent since Apr 17**. Encampment at the vacant SE-corner lot was resolved in late April. Look for: what changed? Fencing? Cleared lot? Is the street actually clean? (RT-01/02/03)

**Known findings (from the data):** 115 reports Jan–Jun · median gap 0.4 d · LSD-serviced 100% · gone-on-arrival 12% · neighborhood: WILSHIRE CENTER - KOREATOWN NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?
- Related tracker/findings: RT-01

In [14]:
stop_map(A.iloc[0], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A2. **350 N Avenue 21 — Lincoln Heights** · 78 · ~1.0 day

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.078887,-118.222470)

**Why we're here:** Encampment-adjacent (LSD), refills daily, near river/rail. 0% gone-on-arrival — piles always there when crews arrive.

**Known findings (from the data):** 78 reports Jan–Jun · median gap 1.0 d · LSD-serviced 100% · gone-on-arrival 0% · neighborhood: LINCOLN HEIGHTS NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [15]:
stop_map(A.iloc[1], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A3. **17900 Sherman Way — Reseda** · 78 · ~1.3 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.200925,-118.522869)

**Why we're here:** Valley anchor of the Sherman Way corridor (see Route). Mixed servicing (27% LSD), 27% gone-on-arrival.

**Known findings (from the data):** 78 reports Jan–Jun · median gap 1.3 d · LSD-serviced 27% · gone-on-arrival 27% · neighborhood: RESEDA NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [16]:
stop_map(A.iloc[2], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A4. **8142 Pershing Dr — Westchester/Playa** · 76 · same-day gaps

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=33.959027,-118.443930)

**Why we're here:** The oddball: 76 reports in a *low-dumping district*, burst re-reporting (median gap 0 days). One angry building? One chronic pile? Worth solving.

**Known findings (from the data):** 76 reports Jan–Jun · median gap 0.0 d · LSD-serviced 0% · gone-on-arrival 18% · neighborhood: WESTCHESTER/PLAYA NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [17]:
stop_map(A.iloc[3], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A5. **1431 N Hobart Blvd — East Hollywood** · 73 · ~1.1 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.097350,-118.306001)

**Why we're here:** Dense-residential refill corner, no LSD. 27% gone-on-arrival.

**Known findings (from the data):** 73 reports Jan–Jun · median gap 1.1 d · LSD-serviced 0% · gone-on-arrival 27% · neighborhood: EAST HOLLYWOOD NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [18]:
stop_map(A.iloc[4], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A6. **114 S New Hampshire Ave — Koreatown** · 73 · ~0.9 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.072258,-118.292527)

**Why we're here:** **Pair with site #1** (400m apart): the non-encampment control that *stayed active* after #1 went quiet. Same street grid, different mechanism?

**Known findings (from the data):** 73 reports Jan–Jun · median gap 0.9 d · LSD-serviced 0% · gone-on-arrival 14% · neighborhood: WILSHIRE CENTER - KOREATOWN NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [19]:
stop_map(A.iloc[5], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A7. **1342 S Elwood St — Downtown (Fashion District edge)** · 65 · ~0.5 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.025144,-118.235584)

**Why we're here:** Encampment-adjacent, sub-daily re-reports, 8% gone-on-arrival.

**Known findings (from the data):** 65 reports Jan–Jun · median gap 0.5 d · LSD-serviced 100% · gone-on-arrival 8% · neighborhood: DOWNTOWN LOS ANGELES

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [20]:
stop_map(A.iloc[6], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A8. **1201 W 39th Pl — South LA** · 63 · ~2.0 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.014369,-118.297508)

**Why we're here:** Classic residential refill corner in the highest-intensity APC.

**Known findings (from the data):** 63 reports Jan–Jun · median gap 2.0 d · LSD-serviced 0% · gone-on-arrival 14% · neighborhood: EMPOWERMENT CONGRESS NORTH AREA NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [21]:
stop_map(A.iloc[7], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A9. **345 Columbia Ave — Westlake** · 63 · ~1.4 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=34.058439,-118.265332)

**Why we're here:** **32% gone-on-arrival** — highest on this list. Who clears it first?

**Known findings (from the data):** 63 reports Jan–Jun · median gap 1.4 d · LSD-serviced 0% · gone-on-arrival 32% · neighborhood: WESTLAKE NORTH NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [22]:
stop_map(A.iloc[8], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## A10. **Gramercy Pl & Gage Ave — South LA** · 60 · ~2.0 days

[Open in Google Maps](https://www.google.com/maps/search/?api=1&query=33.981850,-118.313378)

**Why we're here:** Intersection refill site; mostly regular LASAN service.

**Known findings (from the data):** 60 reports Jan–Jun · median gap 2.0 d · LSD-serviced 13% · gone-on-arrival 12% · neighborhood: EMPOWERMENT CONGRESS CENTRAL AREA NC

**Research questions:** What appears to be generating the dumping here? Why here instead of nearby? Does the data's explanation seem accurate?

In [23]:
stop_map(A.iloc[9], A, color="crimson")

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

---

# Comparison Sites

Not "nice places" — *controls*. Each combines below-average dumping mix,
few/no chronic sites, and healthy overall 311 use (so people demonstrably
do report — quiet-≠-clean partially controlled). Watch for what's
*different*: alley presence, bin access, frontage upkeep, cameras, signage,
private security/BID crews, renter vs owner housing stock.

## C1. **Studio City — Ventura Blvd (Laurel Canyon ↔ Coldwater)** · 0.59x

[Open area in Google Maps](https://www.google.com/maps/search/?api=1&query=Ventura+Blvd+%26+Laurel+Canyon+Blvd,+Studio+City,+CA)

**Why we're here:** citywide mix, **zero** chronic sites. Primary control; one NC boundary from Van Nuys. Covered by Route stops 8–9.

**Known findings (from the data):** see the clean-comparison table in notebook 06 — below-mix dumping share, few or no chronic places, healthy overall 311 engagement.

**Research questions:** What is *different* here — alleys, bin access, frontage upkeep, cameras, BID/private crews, housing stock? Would the chronic-site mechanisms we see elsewhere even have room to operate?

In [24]:
poly = bounds[bounds["NAME"] == "STUDIO CITY NC"]
m = folium.Map(tiles="cartodbpositron")
folium.GeoJson(poly.to_json(), style_function=lambda f: {"color": "seagreen", "weight": 2,
               "fillColor": "seagreen", "fillOpacity": 0.2},
               tooltip=folium.GeoJsonTooltip(fields=["NAME"])).add_to(m)
m.fit_bounds([[poly.total_bounds[1], poly.total_bounds[0]],
              [poly.total_bounds[3], poly.total_bounds[2]]])
m

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## C2. **Sherman Oaks — Ventura corridor** · 0.82x, low intensity. The gradient

[Open area in Google Maps](https://www.google.com/maps/search/?api=1&query=Ventura+Blvd+%26+Van+Nuys+Blvd,+Sherman+Oaks,+CA)

**Why we're here:** case: same boulevard as Studio City, 2km from Van Nuys's worst blocks.

**Known findings (from the data):** see the clean-comparison table in notebook 06 — below-mix dumping share, few or no chronic places, healthy overall 311 engagement.

**Research questions:** What is *different* here — alleys, bin access, frontage upkeep, cameras, BID/private crews, housing stock? Would the chronic-site mechanisms we see elsewhere even have room to operate?

In [25]:
poly = bounds[bounds["NAME"] == "SHERMAN OAKS NC"]
m = folium.Map(tiles="cartodbpositron")
folium.GeoJson(poly.to_json(), style_function=lambda f: {"color": "seagreen", "weight": 2,
               "fillColor": "seagreen", "fillOpacity": 0.2},
               tooltip=folium.GeoJsonTooltip(fields=["NAME"])).add_to(m)
m.fit_bounds([[poly.total_bounds[1], poly.total_bounds[0]],
              [poly.total_bounds[3], poly.total_bounds[2]]])
m

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## C3. **Woodland Hills–Warner Center** · 12.5% dumping share on high total 311

[Open area in Google Maps](https://www.google.com/maps/search/?api=1&query=Warner+Center,+Woodland+Hills,+CA)

**Why we're here:** engagement (13.4K cases — heavy reporters, few dump reports). Best commercial-corridor control. West of Reseda; pairs with Route stop 1 day.

**Known findings (from the data):** see the clean-comparison table in notebook 06 — below-mix dumping share, few or no chronic places, healthy overall 311 engagement.

**Research questions:** What is *different* here — alleys, bin access, frontage upkeep, cameras, BID/private crews, housing stock? Would the chronic-site mechanisms we see elsewhere even have room to operate?

In [26]:
poly = bounds[bounds["NAME"] == "WOODLAND HILLS-WARNER CENTER NC"]
m = folium.Map(tiles="cartodbpositron")
folium.GeoJson(poly.to_json(), style_function=lambda f: {"color": "seagreen", "weight": 2,
               "fillColor": "seagreen", "fillOpacity": 0.2},
               tooltip=folium.GeoJsonTooltip(fields=["NAME"])).add_to(m)
m.fit_bounds([[poly.total_bounds[1], poly.total_bounds[0]],
              [poly.total_bounds[3], poly.total_bounds[2]]])
m

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## C4. **Granada Hills North** · 11.6% share. "Ordinary but clean" single-family

[Open area in Google Maps](https://www.google.com/maps/search/?api=1&query=Granada+Hills,+Los+Angeles,+CA)

**Why we're here:** Valley control — demographically closer to Reseda than Porter Ranch is.

**Known findings (from the data):** see the clean-comparison table in notebook 06 — below-mix dumping share, few or no chronic places, healthy overall 311 engagement.

**Research questions:** What is *different* here — alleys, bin access, frontage upkeep, cameras, BID/private crews, housing stock? Would the chronic-site mechanisms we see elsewhere even have room to operate?

In [27]:
poly = bounds[bounds["NAME"] == "GRANADA HILLS NORTH NC"]
m = folium.Map(tiles="cartodbpositron")
folium.GeoJson(poly.to_json(), style_function=lambda f: {"color": "seagreen", "weight": 2,
               "fillColor": "seagreen", "fillOpacity": 0.2},
               tooltip=folium.GeoJsonTooltip(fields=["NAME"])).add_to(m)
m.fit_bounds([[poly.total_bounds[1], poly.total_bounds[0]],
              [poly.total_bounds[3], poly.total_bounds[2]]])
m

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

## C5. **Porter Ranch** · 5.2% share

[Open area in Google Maps](https://www.google.com/maps/search/?api=1&query=Porter+Ranch,+Los+Angeles,+CA)

**Why we're here:** citywide minimum among engaged NCs. The near-zero anchor; note how much built form (HOA, no alleys, no aging commercial strip) explains before any city service does.

**Known findings (from the data):** see the clean-comparison table in notebook 06 — below-mix dumping share, few or no chronic places, healthy overall 311 engagement.

**Research questions:** What is *different* here — alleys, bin access, frontage upkeep, cameras, BID/private crews, housing stock? Would the chronic-site mechanisms we see elsewhere even have room to operate?

In [28]:
poly = bounds[bounds["NAME"] == "PORTER RANCH NC"]
m = folium.Map(tiles="cartodbpositron")
folium.GeoJson(poly.to_json(), style_function=lambda f: {"color": "seagreen", "weight": 2,
               "fillColor": "seagreen", "fillOpacity": 0.2},
               tooltip=folium.GeoJsonTooltip(fields=["NAME"])).add_to(m)
m.fit_bounds([[poly.total_bounds[1], poly.total_bounds[0]],
              [poly.total_bounds[3], poly.total_bounds[2]]])
m

**Standardized checklist** — fill in during/after the visit:

- **Generating it?**
- **Why here, not nearby?**
- **Data's explanation accurate?** (yes / partly / no):
- **Site features** (keep what applies): alley, vacant lot, apartments, commercial, industrial, loading dock, encampment, bus stop, freeway ramp, parking lot, retaining wall, fence/blank wall, tree well/landscaping, dumping signs, public trash cans, bulky-pickup signage
- **Lighting** (good/fair/poor): · **Camera** (y/n): · **Visibility** (hidden/mixed/high):
- **Vehicle access** (easy/hard): · **Truck access** (easy/hard):
- **Evidence of repeated cleanup** (y/n): · **repeated dumping** (y/n):
- **Worth interviewing** (business/resident/city worker/BID/security/other):
- **Unexpected / surprising / new research question?**

**Observations:**

*(type here during/after the visit)*

---

After the field day: copy filled sections into dated files in `field-notes/`,
promote durable answers to `docs/research_tracker.md` and
`docs/observatory_findings.md`, and bring surprises to the next analysis
notebook.